In [42]:
from pathlib import Path
import sys

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.features import add_features

In [2]:
RANDOM_STATE = 42

# Загрузка данных

In [3]:
train_data = pd.read_csv(project_root / "input_files/train.csv")
test_data = pd.read_csv(project_root / "input_files/test.csv")

# Бейзлайн

In [4]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


берем признаки Pclass, Sex, Age, SibSp, Parch, Fare, Embarked

In [5]:
num_features = ['Age', 'SibSp', 'Parch', 'Fare']
cat_features = ['Sex', 'Pclass', 'Embarked']

features = num_features + cat_features

X_train = train_data[features]
y_train = train_data['Survived']

In [6]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

transformer = ColumnTransformer([
    ('num_pipeline', num_pipe, num_features),
    ('cat_pipeline', cat_pipe, cat_features),
])

model = Pipeline([
    ('preprocessing', transformer),
    ('lr', LogisticRegression())
])

In [7]:
cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

cv = cross_val_score(
    estimator=model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring='accuracy',
    error_score='raise'
)

In [8]:
print(f'CV scores: {cv}')
print(f'Mean accuracy: {cv.mean():.4f}')
print(f'Std accuracy: {cv.std():.4f}')

CV scores: [0.78212291 0.80337079 0.79775281 0.78089888 0.82022472]
Mean accuracy: 0.7969
Std accuracy: 0.0146


Сравним с классификатором, который предсказывает самый частый класс

In [9]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent")

cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

dummy_cv = cross_val_score(
    DummyClassifier(strategy='most_frequent'),
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy'
)

print(f'Dummy accuracy: {dummy_cv.mean():.4f}')

Dummy accuracy: 0.6162


# Эксперименты с применением feature engineering

In [10]:
def evaluate_features(X, num_features, cat_features, y, n_splits=5, random_state=RANDOM_STATE, shuffle=True):
    cv = StratifiedKFold(n_splits=n_splits, random_state=RANDOM_STATE, shuffle=shuffle)

    transformer = ColumnTransformer([
        ('num', num_pipe, num_features),
        ('cat', cat_pipe, cat_features)
    ])

    model = Pipeline([
        ('preprocessing', transformer),
        ('lr', LogisticRegression())
    ])

    scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='accuracy'
    )

    return scores.mean(), scores.std()

### E1: исходные фичи + IsChild

In [11]:
X_exp = X_train.copy()

X_exp['IsChild'] = np.where(
    X_exp['Age'].isna(),
    np.nan,
    (X_exp['Age'] <= 12).astype(int)
)

num_features = ['Age', 'SibSp', 'Parch', 'Fare', 'IsChild']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8103257799259305), np.float64(0.008972205733988832))


### E2: объединяем SibSp и Parch в FamilySize

In [12]:
X_exp = X_train.copy()

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

num_features = ['Age', 'Fare', 'FamilySize']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8013495700207145), np.float64(0.01094886536642612))


### E3: исходные + IsAlone

In [13]:
X_exp = X_train.copy()

X_exp['IsAlone'] = ((X_exp['SibSp'] + X_exp['Parch']) == 0).astype(int)

num_features = ['Age', 'Fare', 'Parch', 'SibSp', 'IsAlone']
cat_features = ['Sex', 'Pclass', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.7991274872889335), np.float64(0.018284398787139593))


### E4: + размер семьи (малый, средний, большой)

In [14]:
X_exp = X_train.copy()

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['SibSp', 'Parch', 'FamilySize'])

num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Pclass', 'Embarked', 'FamilyGroup']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8114556525014123), np.float64(0.011426225981788643))


### E5: извлекаем титул из имени

In [15]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [16]:
X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mile': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

In [17]:
num_features = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8282844768062269), np.float64(0.013075641273705182))


### E6: Title + IsChild

In [18]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['IsChild'] = np.where(
    X_exp['Age'].isna(),
    np.nan,
    (X_exp['Age'] <= 12).astype(int)
)

num_features = ['Age', 'Fare', 'SibSp', 'Parch', 'IsChild']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8271608813006089), np.float64(0.008972851263963317))


одновременное использование Title и IsChild не улучшает метрику. Это я бы объяснил тем, что Title уже содержит в себе информацию о том, ребенок это или нет (класс Master -- обращение к мальчику)

### E7: Title + FamilyGroup

In [19]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['SibSp', 'Parch', 'FamilySize'])

num_features = ['Age', 'Fare']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8282656455966355), np.float64(0.011245216054438085))


## E8: добавим признак HasCabin

In [20]:
X_exp = train_data.copy()

X_exp['Title'] = (
    X_exp['Name']
    .str.extract(r',\s*([^.]*)\.')[0]
    .str.strip()
)

X_exp['Title'] = X_exp['Title'].replace({
    'Mme': 'Mrs',
    'Mlle': 'Miss',
    'Ms': 'Miss',
})

common_titles = ['Mr', 'Miss', 'Mrs', 'Master']

X_exp['Title'] = X_exp['Title'].where(
    X_exp['Title'].isin(common_titles),
    'Rare',
)

X_exp['FamilySize'] = X_exp['SibSp'] + X_exp['Parch'] + 1

X_exp['FamilyGroup'] = pd.cut(
    X_exp['FamilySize'],
    bins=[0, 1, 4, float('inf')],
    labels=['Alone', 'Small', 'Large']
)

X_exp = X_exp.drop(columns=['FamilySize'])

X_exp['HasCabin'] = X_exp['Cabin'].notna().astype(int)

num_features = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup', 'HasCabin']

print(evaluate_features(X_exp, num_features, cat_features, y_train))

(np.float64(0.8305128366078716), np.float64(0.0068841858906640305))


лучший результат

## LightGBM

In [28]:
X_train = add_features(train_data)
y_train = train_data['Survived']


num_features = ['Age', 'Fare', 'SibSp', 'Parch', 'HasCabin']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup']

In [30]:
for col in cat_features:
    X_train[col] = X_train[col].astype("category")

X_lgbm = X_train[num_features+cat_features]

cv = StratifiedKFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)

In [33]:
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        'max_depth': trial.suggest_int("max_depth", 2, 10),
        'num_leaves': trial.suggest_int("num_leaves", 2, 31),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        'min_child_samples': trial.suggest_int("min_child_samples", 5, 40)
    }

    lgbm = LGBMClassifier(
        **params,
        random_state=42, 
        verbosity=-1
    )

    cv_scores = cross_val_score(
        lgbm,
        X_lgbm,
        y_train,
        cv=cv,
        scoring='accuracy',
    )

    return cv_scores.mean()

In [34]:
study_lgbm = optuna.create_study(
    study_name="LGBM hyperparameters search", 
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=50)

[I 2026-10-27 21:50:54,262] A new study created in memory with name: LGBM hyperparameters search
[I 2026-10-27 21:50:55,799] Trial 0 finished with value: 0.8181595631159375 and parameters: {'n_estimators': 437, 'max_depth': 10, 'num_leaves': 23, 'learning_rate': 0.03968793330444373, 'min_child_samples': 10}. Best is trial 0 with value: 0.8181595631159375.
[I 2026-10-27 21:50:55,994] Trial 1 finished with value: 0.8316301550436256 and parameters: {'n_estimators': 240, 'max_depth': 2, 'num_leaves': 27, 'learning_rate': 0.039913058785616795, 'min_child_samples': 30}. Best is trial 1 with value: 0.8316301550436256.
[I 2026-10-27 21:50:56,323] Trial 2 finished with value: 0.828253091456908 and parameters: {'n_estimators': 118, 'max_depth': 10, 'num_leaves': 26, 'learning_rate': 0.016305687346221478, 'min_child_samples': 11}. Best is trial 1 with value: 0.8316301550436256.
[I 2026-10-27 21:50:56,653] Trial 3 finished with value: 0.8349883874207521 and parameters: {'n_estimators': 265, 'max_d

In [35]:
# Получаем лучшее испытание
best_trial_lgbm = study_lgbm.best_trial

# Выводим результаты
print("Лучшее значение метрики:", best_trial_lgbm.value)
print("Соответствующие гиперпараметры:")
for param_name, param_value in best_trial_lgbm.params.items():
    print(f"  {param_name}: {param_value}")

Лучшее значение метрики: 0.8495951289937856
Соответствующие гиперпараметры:
  n_estimators: 683
  max_depth: 2
  num_leaves: 8
  learning_rate: 0.078690170029021
  min_child_samples: 9


## XGBoost

In [36]:
X_train = add_features(train_data)
y_train = train_data['Survived']


num_features = ['Age', 'Fare', 'SibSp', 'Parch']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup', 'HasCabin']

In [37]:
for col in cat_features:
    X_train[col] = X_train[col].astype("category")

X_xgb = X_train[num_features+cat_features]

cv = StratifiedKFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)

In [38]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }

    xgb = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )

    cv_scores = cross_val_score(
        xgb,
        X_xgb,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise",
    )

    return cv_scores.mean()

In [39]:
study_xgb = optuna.create_study(
    study_name="XGB hyperparameters search",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_xgb.optimize(objective_xgb, n_trials=50)

print(f"Best CV Accuracy: {study_xgb.best_value:.4f}")
print(f"Best parameters: {study_xgb.best_params}")

[I 2026-10-27 21:54:49,860] A new study created in memory with name: XGB hyperparameters search
[I 2026-10-27 21:54:54,530] Trial 0 finished with value: 0.8417362375243236 and parameters: {'n_estimators': 437, 'max_depth': 8, 'min_child_weight': 7.587945476302646, 'learning_rate': 0.07661100707771368, 'subsample': 0.6624074561769746}. Best is trial 0 with value: 0.8417362375243236.
[I 2026-10-27 21:54:57,593] Trial 1 finished with value: 0.8394890465130876 and parameters: {'n_estimators': 240, 'max_depth': 2, 'min_child_weight': 8.795585311974417, 'learning_rate': 0.07725378389307355, 'subsample': 0.8832290311184181}. Best is trial 0 with value: 0.8417362375243236.
[I 2026-10-27 21:55:00,515] Trial 2 finished with value: 0.8249199673592369 and parameters: {'n_estimators': 118, 'max_depth': 8, 'min_child_weight': 8.491983767203795, 'learning_rate': 0.020589728197687916, 'subsample': 0.6727299868828402}. Best is trial 0 with value: 0.8417362375243236.
[I 2026-10-27 21:55:03,873] Trial 3 

Best CV Accuracy: 0.8529
Best parameters: {'n_estimators': 516, 'max_depth': 7, 'min_child_weight': 6.014435154132807, 'learning_rate': 0.07740750704748588, 'subsample': 0.712096521064806}


## Catboost

In [50]:
X_train = add_features(train_data)
y_train = train_data['Survived']


num_features = ['Age', 'Fare', 'SibSp', 'Parch', 'HasCabin']
cat_features = ['Sex', 'Pclass', 'Title', 'Embarked', 'FamilyGroup']

In [51]:
X_cb = X_train[num_features+cat_features]

for col in cat_features:
    X_cb[col] = X_cb[col].astype("string").fillna("Missing")

cv = StratifiedKFold(n_splits=5, random_state=RANDOM_STATE, shuffle=True)

In [47]:
X_cb.dtypes

Age             float64
Fare            float64
SibSp             int64
Parch             int64
HasCabin          int64
Sex            category
Pclass         category
Title          category
Embarked       category
FamilyGroup    category
dtype: object

In [52]:
cb = CatBoostClassifier(
    random_state=42,
)

cv_scores = cross_val_score(
    cb,
    X_cb,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    error_score="raise",
    params={"cat_features": cat_features}
)

In [53]:
print(cv_scores.mean(), cv_scores.std())

0.8237712635741637 0.012880556537491605


In [57]:
def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 2, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 1, 10, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 2.0
        ),
    }

    cb = CatBoostClassifier(
        **params,
        random_state=42,
    )

    cv_scores = cross_val_score(
        cb,
        X_cb,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise",
        params={"cat_features": cat_features}
    )

    return cv_scores.mean()

In [58]:
study_cb = optuna.create_study(
    study_name="CatBoost hyperparameters search",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_cb.optimize(objective_catboost, n_trials=50)

print(f"Best CV Accuracy: {study_cb.best_value:.4f}")
print(f"Best parameters: {study_cb.best_params}")

[I 2026-10-27 22:25:40,329] A new study created in memory with name: CatBoost hyperparameters search
[I 2026-10-27 22:26:46,640] Trial 0 finished with value: 0.8271483271608814 and parameters: {'iterations': 500, 'depth': 8, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 3, 'random_strength': 0.31203728088487304}. Best is trial 0 with value: 0.8271483271608814.
[I 2026-10-27 22:26:56,674] Trial 1 finished with value: 0.8338710689849979 and parameters: {'iterations': 324, 'depth': 2, 'learning_rate': 0.13394334706750485, 'l2_leaf_reg': 3, 'random_strength': 1.416145155592091}. Best is trial 1 with value: 0.8338710689849979.
[I 2026-10-27 22:27:21,253] Trial 2 finished with value: 0.8316364321134895 and parameters: {'iterations': 216, 'depth': 8, 'learning_rate': 0.12106896936002161, 'l2_leaf_reg': 1, 'random_strength': 0.36364993441420124}. Best is trial 1 with value: 0.8338710689849979.
[I 2026-10-27 22:27:37,952] Trial 3 finished with value: 0.8293829640323898 and parameters: {'

Best CV Accuracy: 0.8440
Best parameters: {'iterations': 743, 'depth': 4, 'learning_rate': 0.06602784138887151, 'l2_leaf_reg': 8, 'random_strength': 0.6243934569471328}
